# DGX 25MCSA19 Export And Cleanup

Upload/run this notebook from outside the project folder if possible. It operates on:

`/workspace/temp/25mcsa19/working/temp/25mcsa19`

It creates a ZIP archive first, verifies it, then optionally deletes Kaggle credentials and the GeoDiff Jupyter kernel files.

Default archive mode excludes raw datasets, downloads, caches, venvs, and secrets. This keeps the backup small and avoids storing credentials.

## 1. Settings And Safety Guard

In [ ]:
from pathlib import Path
import fnmatch
import hashlib
import json
import os
import shutil
import subprocess
import sys
import time
import zipfile

ASSIGNED_ROOT = Path('/workspace/temp/25mcsa19/working/temp/25mcsa19').resolve()
ALLOWED_PARENT = Path('/workspace/temp/25mcsa19/working/temp').resolve()

# Archive is intentionally outside ASSIGNED_ROOT so it is not included inside itself.
EXPORT_DIR = ALLOWED_PARENT / 'geodiff_25mcsa19_exports'
ARCHIVE_NAME = f'geodiff_25mcsa19_backup_{time.strftime("%Y%m%d_%H%M%S")}.zip'
ARCHIVE_PATH = EXPORT_DIR / ARCHIVE_NAME

# Keep this False unless you deliberately want raw SAFE/dataset/download files in the zip.
INCLUDE_RAW_DATASETS = False
INCLUDE_VENV_AND_CACHE = False
INCLUDE_SECRETS = False

# Cleanup is separated from export and requires explicit confirmation in the cleanup cell.
DELETE_KAGGLE_CREDENTIALS = True
DELETE_LOCAL_KERNELSPEC = True
DELETE_USER_KERNELSPEC = True
DELETE_VENV = False  # leave False if this notebook is running from the GeoDiff venv/kernel

KERNEL_NAME = 'geodiff-py311-25mcsa19'

def assert_under(path, root):
    path = Path(path).resolve()
    root = Path(root).resolve()
    if path != root and root not in path.parents:
        raise RuntimeError(f'Refusing path outside {root}: {path}')
    return path

if not ASSIGNED_ROOT.exists():
    raise FileNotFoundError(f'Assigned root not found: {ASSIGNED_ROOT}')
assert_under(ASSIGNED_ROOT, ALLOWED_PARENT)
EXPORT_DIR.mkdir(parents=True, exist_ok=True)
assert_under(EXPORT_DIR, ALLOWED_PARENT)

print('Python:', sys.executable)
print('Assigned root:', ASSIGNED_ROOT)
print('Archive path:', ARCHIVE_PATH)
print('Include raw datasets:', INCLUDE_RAW_DATASETS)
print('Include venv/cache:', INCLUDE_VENV_AND_CACHE)
print('Include secrets:', INCLUDE_SECRETS)

## 2. Inspect Folder Size

In [ ]:
def run(command, check=False):
    command = [str(value) for value in command]
    print('+', ' '.join(command), flush=True)
    return subprocess.run(command, check=check)

run(['du', '-sh', ASSIGNED_ROOT])
for child in sorted(ASSIGNED_ROOT.iterdir()):
    run(['du', '-sh', child])

## 3. Create ZIP Archive

    This archive excludes credentials and heavy temporary folders by default. Change settings above if needed before running this cell.

In [ ]:
ALWAYS_EXCLUDE_DIR_NAMES = {
    '__pycache__', '.git', '.ipynb_checkpoints'
}
SECRET_DIR_NAMES = {'secrets', '.kaggle'}
VENV_CACHE_DIR_NAMES = {'.venvs', '.cache', 'jupyter_kernels'}
RAW_DATA_DIR_NAMES = {'datasets', 'downloads', 'sota_sources'}

EXCLUDE_FILE_PATTERNS = [
    '*.tmp', '*.temp', '*.lock', '*.part',
    'kaggle.json', 'kaggle.netrc', '.netrc',
]
if not INCLUDE_RAW_DATASETS:
    EXCLUDE_FILE_PATTERNS.extend(['*.SAFE.zip', '*.zip', '*.jp2'])
if not INCLUDE_SECRETS:
    EXCLUDE_FILE_PATTERNS.extend(['*.token', '*token*'])

def should_exclude(path):
    rel = path.relative_to(ASSIGNED_ROOT)
    parts = set(rel.parts)
    if parts & ALWAYS_EXCLUDE_DIR_NAMES:
        return True
    if not INCLUDE_SECRETS and parts & SECRET_DIR_NAMES:
        return True
    if not INCLUDE_VENV_AND_CACHE and parts & VENV_CACHE_DIR_NAMES:
        return True
    if not INCLUDE_RAW_DATASETS and parts & RAW_DATA_DIR_NAMES:
        return True
    name = path.name
    return any(fnmatch.fnmatch(name, pattern) for pattern in EXCLUDE_FILE_PATTERNS)

if ARCHIVE_PATH.exists():
    raise FileExistsError(f'Archive already exists: {ARCHIVE_PATH}')

file_count = 0
total_bytes = 0
started = time.time()
with zipfile.ZipFile(ARCHIVE_PATH, 'w', compression=zipfile.ZIP_DEFLATED, compresslevel=6) as zf:
    for path in ASSIGNED_ROOT.rglob('*'):
        if should_exclude(path):
            continue
        if path.is_file():
            rel = path.relative_to(ASSIGNED_ROOT)
            zf.write(path, arcname=str(rel))
            file_count += 1
            total_bytes += path.stat().st_size
            if file_count % 1000 == 0:
                print(f'Archived {file_count:,} files, source bytes={total_bytes/1024**3:.2f} GiB')

elapsed = time.time() - started
print('Archive created:', ARCHIVE_PATH)
print('Files archived:', f'{file_count:,}')
print('Source bytes:', f'{total_bytes/1024**3:.2f} GiB')
print('Archive size:', f'{ARCHIVE_PATH.stat().st_size/1024**3:.2f} GiB')
print('Elapsed:', f'{elapsed/60:.1f} min')

## 4. Verify ZIP And SHA256

In [ ]:
if not ARCHIVE_PATH.exists():
    raise FileNotFoundError(ARCHIVE_PATH)

print('Testing zip integrity...')
with zipfile.ZipFile(ARCHIVE_PATH, 'r') as zf:
    bad = zf.testzip()
    members = zf.namelist()
if bad is not None:
    raise RuntimeError(f'Corrupt zip member: {bad}')

print('Zip OK. Members:', len(members))
print('First 30 members:')
for name in members[:30]:
    print(' ', name)

print('Computing SHA256...')
sha = hashlib.sha256()
with ARCHIVE_PATH.open('rb') as handle:
    for chunk in iter(lambda: handle.read(1024 * 1024 * 16), b''):
        sha.update(chunk)
sha_path = ARCHIVE_PATH.with_suffix(ARCHIVE_PATH.suffix + '.sha256')
sha_text = f'{sha.hexdigest()}  {ARCHIVE_PATH.name}\n'
sha_path.write_text(sha_text, encoding='utf-8')
print(sha_text)
print('SHA file:', sha_path)

## 5. Cleanup Kaggle Credentials And GeoDiff Kernel

    Run this only after the ZIP integrity check succeeds and you have noted the archive path.

    The cell deletes only exact known paths:

    - assigned-root Kaggle credentials
    - local assigned-root kernelspec
    - optional user Jupyter kernelspec named `geodiff-py311-25mcsa19`
    - optional GeoDiff venv if `DELETE_VENV=True`

In [ ]:
CONFIRM_CLEANUP = ''  # set to 'DELETE' after verifying archive and SHA256

if CONFIRM_CLEANUP != 'DELETE':
    raise RuntimeError("Set CONFIRM_CLEANUP = 'DELETE' in this cell before cleanup.")

if not ARCHIVE_PATH.exists():
    raise FileNotFoundError(f'Archive missing, cleanup refused: {ARCHIVE_PATH}')

deleted = []
missing = []

def remove_file(path):
    path = Path(path).resolve()
    if path.exists():
        if path.is_file() or path.is_symlink():
            path.unlink()
            deleted.append(str(path))
        else:
            raise RuntimeError(f'Expected file, got directory: {path}')
    else:
        missing.append(str(path))

def remove_dir(path, allowed_root=None):
    path = Path(path).resolve()
    if allowed_root is not None:
        assert_under(path, allowed_root)
    if path.exists():
        if path.is_dir():
            shutil.rmtree(path)
            deleted.append(str(path))
        else:
            raise RuntimeError(f'Expected directory, got file: {path}')
    else:
        missing.append(str(path))

if DELETE_KAGGLE_CREDENTIALS:
    credential_paths = [
        ASSIGNED_ROOT / 'secrets' / 'kaggle' / 'kaggle.json',
        ASSIGNED_ROOT / 'secrets' / 'kaggle' / 'kaggle.netrc',
        ASSIGNED_ROOT / '.kaggle' / 'kaggle.json',
        ASSIGNED_ROOT / '.kaggle' / 'kaggle.netrc',
        ASSIGNED_ROOT / 'kaggle.json',
        ASSIGNED_ROOT / 'kaggle.netrc',
    ]
    for path in credential_paths:
        assert_under(path, ASSIGNED_ROOT)
        remove_file(path)
    for directory in [ASSIGNED_ROOT / 'secrets' / 'kaggle', ASSIGNED_ROOT / '.kaggle']:
        if directory.exists() and directory.is_dir() and not any(directory.iterdir()):
            directory.rmdir()
            deleted.append(str(directory))

if DELETE_LOCAL_KERNELSPEC:
    remove_dir(ASSIGNED_ROOT / 'jupyter_kernels' / KERNEL_NAME, allowed_root=ASSIGNED_ROOT)

if DELETE_USER_KERNELSPEC:
    # This is outside ASSIGNED_ROOT by design. It deletes only the exact kernel name.
    user_kernel = Path.home() / '.local' / 'share' / 'jupyter' / 'kernels' / KERNEL_NAME
    expected_suffix = Path('.local/share/jupyter/kernels') / KERNEL_NAME
    if user_kernel.parts[-len(expected_suffix.parts):] != expected_suffix.parts:
        raise RuntimeError(f'Unexpected user kernel path: {user_kernel}')
    remove_dir(user_kernel)

if DELETE_VENV:
    # Do not enable this while running from the GeoDiff venv/kernel.
    current_python = Path(sys.executable).resolve()
    venv_dir = ASSIGNED_ROOT / '.venvs' / 'geodiff-py311'
    if str(current_python).startswith(str(venv_dir.resolve())):
        raise RuntimeError(f'Current Python is inside venv; switch kernels before deleting: {current_python}')
    remove_dir(venv_dir, allowed_root=ASSIGNED_ROOT)

cleanup_report = {
    'archive': str(ARCHIVE_PATH),
    'deleted': deleted,
    'missing': missing,
    'timestamp': time.strftime('%Y-%m-%d %H:%M:%S'),
}
report_path = EXPORT_DIR / f'cleanup_report_{time.strftime("%Y%m%d_%H%M%S")}.json'
report_path.write_text(json.dumps(cleanup_report, indent=2), encoding='utf-8')

print('Deleted paths:')
for path in deleted:
    print(' -', path)
print('Missing/skipped paths:')
for path in missing:
    print(' -', path)
print('Cleanup report:', report_path)
print('Archive still present:', ARCHIVE_PATH, ARCHIVE_PATH.exists())

## 6. Optional Final Check

In [ ]:
print('Archive exists:', ARCHIVE_PATH.exists(), ARCHIVE_PATH)
print('SHA exists:', ARCHIVE_PATH.with_suffix(ARCHIVE_PATH.suffix + '.sha256').exists())
print('Kaggle json exists:', (ASSIGNED_ROOT / 'secrets' / 'kaggle' / 'kaggle.json').exists())
print('Local kernelspec exists:', (ASSIGNED_ROOT / 'jupyter_kernels' / KERNEL_NAME).exists())
print('User kernelspec exists:', (Path.home() / '.local' / 'share' / 'jupyter' / 'kernels' / KERNEL_NAME).exists())